# 02.04 声音复刻（Voice Cloning）

## 本节概述

<table style="text-align: left; margin-left: 0;">
<tr><td align="left"><b>前置要求</b></td><td align="left">已完成 02.03 ASR/TTS</td></tr>
<tr><td align="left"><b>本节目标</b></td><td align="left">用华为云 SIS 克隆自定义音色并合成语音</td></tr>
<tr><td align="left"><b>本节内容</b></td><td align="left">注册音色 → 查询确认 → 流式合成（ONLY/MULTI 模式）</td></tr>
</table>


In [ ]:
# ===== 确保参考音频存在（不存在则从 ModelScope 下载）=====
import os
for fname in ["./images/clone_example.wav", "./images/clone_example_16k.wav"]:
    if not os.path.exists(fname):
        print(f"⏳ 下载参考音频 {os.path.basename(fname)} ...")
        os.makedirs("./images", exist_ok=True)
        try:
            import modelscope
        except ImportError:
            import subprocess
            subprocess.check_call(['pip', 'install', 'modelscope', '-q'])
        from modelscope.hub.snapshot_download import snapshot_download
        snapshot_download('Kumako/speech_audio_samples', repo_type='dataset', local_dir='./images_tmp')
        import shutil
        for item in os.listdir('./images_tmp'):
            s = os.path.join('./images_tmp', item)
            d = os.path.join('./images', item)
            if not os.path.exists(d):
                shutil.move(s, d)
        shutil.rmtree('./images_tmp', ignore_errors=True)
        print("✅ 参考音频已就绪")
        break
else:
    print("✅ 本地已有参考音频")

# 💡 如果 .env 尚未创建，请先运行 02.02 的第一个 cell 创建并填入凭证
import os, sys, base64, time
sys.path.insert(0, os.path.abspath("./src"))
from dotenv import load_dotenv
load_dotenv()

from huaweicloud_sis.bean.vcs_register_voice_request import RegisterVoiceRequest
from huaweicloud_sis.client.vcs_client import VcsClient
from huaweicloud_sis.bean.sis_config import SisConfig

ak = os.getenv('HUAWEI_SIS_AK', '')
sk = os.getenv('HUAWEI_SIS_SK', '')
region = os.getenv('HUAWEI_SIS_REGION', 'cn-east-3')
project_id = os.getenv('HUAWEI_SIS_PROJECT_ID', '')
service_endpoint = f"https://sis-ext.{region}.myhuaweicloud.com"


def register_voice(audio_path, voice_name):
    """
    注册自定义音色

    要求:
        - 音频采样率 ≥ 16kHz
        - 音频时长 ≥ 5 秒
        - 内容清晰，无背景噪声
    """
    # 读取音频并 Base64 编码（必须是 base64 字符串）
    with open(audio_path, 'rb') as f:
        data = str(base64.b64encode(f.read()), 'utf-8')

    config = SisConfig()
    config.set_connect_timeout(10)
    config.set_read_timeout(60)
    vcs_client = VcsClient(ak, sk, region, project_id,
                           sis_config=config, service_endpoint=service_endpoint)

    # 构造请求（2个位置参数：base64数据 + 音色名）
    register_request = RegisterVoiceRequest(data, voice_name)
    result = vcs_client.register_voice(register_request)
    return result


def query_voices(limit=50, offset=0):
    """查询已注册的所有音色"""
    config = SisConfig()
    config.set_connect_timeout(10)
    config.set_read_timeout(60)
    vcs_client = VcsClient(ak, sk, region, project_id,
                           sis_config=config, service_endpoint=service_endpoint)
    return vcs_client.query_voice_name(limit, offset)


# 先查询已有音色
print("📋 当前已注册的音色列表:")
try:
    result = query_voices()
    print(result)
except Exception as e:
    print(f"查询失败: {e}")

# 注册新音色（用时间戳避免重名）
voice_name = f"my_voice_{int(time.time())}"
register_audio = './images/clone_example.wav'   # 课程自带的参考音频

print(f"\n准备注册音色: {voice_name}")
print(f"参考音频: {register_audio}")
try:
    result = register_voice(register_audio, voice_name)
    print(f"\n✅ 注册结果: {result}")
    print("💡 记下 voice_name，后续合成时用")
except Exception as e:
    print(f"注册失败: {e}")
    print("💡 请检查：1) 参考音频是否 ≥5 秒 16kHz 2) voice_name 是否重复 3) project_id 是否正确")


注册是**异步**的——提交后华为云需要几秒到几十秒处理。处理完成后才能用于合成。

## 1. 使用克隆音色合成（流式 WebSocket）

声音复刻的合成用 **VcsStreamClient**（WebSocket 流式接口）。关键 API：

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">API</th><th align="left">正确用法</th></tr>
<tr><td align="left"><code>VcsStreamCallBack</code></td><td align="left">回调基类，子类实现 on_response(data) 等方法，data 是原始音频 bytes</td></tr>
<tr><td align="left"><code>VcsStreamClient</code></td><td align="left"><code>(ak, sk, use_aksk=True, region, project_id, callback, config)</code></td></tr>
<tr><td align="left"><code>VcsStreamRequest</code></td><td align="left">无参构造，用 setter 设置参数</td></tr>
<tr><td align="left">ONLY 模式</td><td align="left"><code>set_text(text)</code> + <code>vcs_client.synthesis(request)</code></td></tr>
<tr><td align="left">MULTI 模式</td><td align="left"><code>sendStart(request)</code> + <code>sendMsg(text)</code> + <code>sendEnd()</code></td></tr>
</table>

> 💡 **注意**：回调的 `on_response(data)` 中 `data` 是**原始音频 bytes**，直接写入文件即可，不需要 base64 解码。

In [ ]:
from huaweicloud_sis.client.vcs_stream_client import VcsStreamClient
from huaweicloud_sis.bean.vcs_stream_request import VcsStreamRequest
from huaweicloud_sis.bean.callback import VcsStreamCallBack


class MyCallback(VcsStreamCallBack):
    """回调类：接收合成音频数据并保存到文件"""

    def __init__(self, save_path):
        self._f = open(save_path, 'wb')
        self._save_path = save_path

    def on_open(self):
        print(f'  🔗 WebSocket 连接成功')

    def on_start(self, message):
        print(f'  ▶️ 开始合成: {message}')

    def on_response(self, data):
        # data 是原始音频 bytes，直接写入（不需要 base64 解码）
        print(f'  📦 接收音频数据: {len(data)} bytes')
        self._f.write(data)

    def on_end(self, message):
        print(f'  ✅ 合成完成: {message}')
        self._f.close()

    def on_close(self):
        print(f'  🔌 WebSocket 已关闭')
        self._f.close()

    def on_error(self, error):
        print(f'  ❌ 合成错误: {error}')
        self._f.close()


def vcs_synthesis(text, voice_name, output_path='./clone_output.wav', mode='ONLY'):
    """
    用克隆音色（或预置音色）合成语音

    参数:
        text: 待合成文本
        voice_name: 音色名称（注册的自定义音色或预置音色）
        output_path: 输出音频路径
        mode: 'ONLY' 单文本模式 / 'MULTI' 多文本模式
    """
    my_callback = MyCallback(output_path)
    config = SisConfig()
    config.set_connect_timeout(10)
    config.set_read_timeout(10)
    config.set_websocket_wait_time(20)

    vcs_client = VcsStreamClient(
        ak=ak, sk=sk, use_aksk=True, region=region,
        project_id=project_id, callback=my_callback, config=config
    )

    vcs_request = VcsStreamRequest()
    vcs_request.set_voice_name(voice_name)
    vcs_request.set_audio_format('mp3')
    vcs_request.set_sample_rate('16000')
    vcs_request.set_volume(50)
    vcs_request.set_speed(0)
    vcs_request.set_pitch(0)
    vcs_request.set_text_pieces(mode)

    if mode == 'ONLY':
        vcs_request.set_text(text)
        vcs_client.synthesis(vcs_request)
    elif mode == 'MULTI':
        vcs_client.sendStart(vcs_request)
        vcs_client.sendMsg(text)
        vcs_client.sendEnd()


# 用克隆音色合成（需先完成 Cell[1] 注册且音色处理完成）
# ⚠️ 流式 VCS 接口只支持注册的克隆音色，不支持预置音色
print("🎭 用克隆音色合成语音...")
try:
    _voice = voice_name  # Cell[1] 注册的音色名
    print(f"  使用音色: {_voice}")
except NameError:
    raise SystemExit("⚠️ 未检测到 voice_name，请先运行 Cell[1] 注册克隆音色。\n"
                     "   注册后等待 10-60 秒异步处理完成，再运行本 cell。")
try:
    vcs_synthesis(
        text='你好，这是我用声音复刻技术合成的语音。',
        voice_name=_voice,
        output_path='./clone_demo.mp3'
    )
    from IPython.display import Audio, display
    display(Audio('./clone_demo.mp3'))
except Exception as e:
    print(f"合成失败（可能音色还在异步处理中，稍后再试）: {e}")
    print("💡 注册后需等待约 30-60 秒，音色处理完成才能用于合成")


## MULTI 模式：模拟大模型实时播报

MULTI 模式可连续发送多段文本实时合成，非常适合对接大模型流式输出（LLM 一边生成文字，VCS 一边播报）：

In [ ]:
import time
from huaweicloud_sis.bean.sis_config import SisConfig
print("🎭 MULTI 模式 - 模拟大模型实时播报:")

# ⚠️ 流式 VCS 接口必须用「注册的克隆音色」，不支持预置音色（如 chinese_xiaoyan_common）
# 复用 Cell[1] 注册、Cell[3] 合成时用的 voice_name
try:
    _voice = voice_name
    print(f"  🎯 使用克隆音色: {_voice}")
except NameError:
    print("⚠️ 未检测到 voice_name，请先运行 Cell[1] 注册音色")
    print("   （注册后需等待约 10-60 秒异步处理完成，才能用于合成）")
    raise SystemExit("请先完成音色注册")

my_callback = MyCallback('./clone_multi.mp3')
config = SisConfig()
config.set_connect_timeout(10); config.set_read_timeout(10); config.set_websocket_wait_time(20)

vcs_client = VcsStreamClient(
    ak=ak, sk=sk, use_aksk=True, region=region,
    project_id=project_id, callback=my_callback, config=config
)

vcs_request = VcsStreamRequest()
vcs_request.set_voice_name(_voice)   # ★ 用克隆音色（流式VCS不支持预置音色）
vcs_request.set_audio_format('mp3')
vcs_request.set_sample_rate('16000')
vcs_request.set_text_pieces('MULTI')

try:
    vcs_client.sendStart(vcs_request)
    sentences = [
        "大家好，我是你们的AI助手。",
        "今天天气真不错，适合出去走走。",
        "如果你有任何问题，随时可以问我。",
    ]
    for i, sent in enumerate(sentences):
        print(f"  📤 发送第 {i+1} 句: {sent}")
        vcs_client.sendMsg(sent)
        time.sleep(0.5)   # 模拟大模型生成延迟
    vcs_client.sendEnd()
    print("✅ MULTI 模式合成完成！")
    print("💡 结果音频保存在 ./clone_multi.mp3")
except Exception as e:
    print(f"❌ 失败: {e}")
    print("💡 可能原因：克隆音色还在异步处理中，请等待 30-60 秒后重试")

### ONLY vs MULTI 模式

<table style="text-align: left; margin-left: 0;">
<tr style="background-color:#f0f0f0">
  <th align="left">模式</th><th align="left">说明</th><th align="left">适用场景</th></tr>
<tr><td align="left"><code>ONLY</code></td><td align="left">发送一段文本，获取一段音频</td><td align="left">合成固定文本</td></tr>
<tr><td align="left"><code>MULTI</code></td><td align="left">连续发送多段文本，实时合成</td><td align="left">大模型流式输出的实时播报</td></tr>
</table>

## 4. 声音复刻的伦理考量

> ⚠️ 声音复刻可克隆任何人的声音，可能带来**电信诈骗、身份冒用**风险。使用时务必：
> 1. 获得**被克隆者的明确授权**；
> 2. 在合成语音中添加**水印**标识；
> 3. 遵守法律法规，不用于欺诈、造谣；
> 4. 平台应做**活体检测**，防止用录音冒充本人。

---

## 本节练习

**练习 1（选择）**：声音复刻相比普通 TTS 的核心优势是？
- A. 合成速度更快
- B. 可以克隆任意参考音频的音色，实现个性化
- C. 音质更高清
- D. 不需要参考音频

**练习 2（填空）**：注册音色对参考音频的要求是：采样率 ≥ ______ kHz，时长 ≥ ______ 秒。注册请求数据必须是 ______ 编码的字符串。

**练习 3（简答）**：VCS 流式合成的回调 `on_response(data)` 中，`data` 是什么格式？需要 base64 解码吗？

> 💡 参考答案见下方 code cell。

In [ ]:
# 查看本节练习答案
!cat ./answer/02.04_voice_cloning/answers.txt
